**As mentioned earlier, the PropBank Release is an ongoing project and includes additional annotations compared to the OntoNotes 5.0 Corpus. Therefore, we need to extract only the existing predicate-argument structures within the standard version of OntoNotes. For this reason, we select the observations from the extracted dataset that exist in the OntoNotes 5.0 Corpus.**

In [ ]:
from nltk.corpus import propbank
import json
import pandas as pd
from urllib.request import urlopen
import numpy as np
import urllib
import os
import re
from os import chdir
from nltk.corpus import treebank
from nltk.tree import Tree
EXPERIMENT_NAME = 'SRL'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/df.csv', na_filter=False)

In [ ]:
del df['Unnamed: 0']

In [ ]:
df

In [ ]:
# Dataset placeholders
data_name_to_google_drive_url = {
    '1': '{{PROPBANK_RELEASE_DRIVE_URL}}',
    '2': '{{ONTONOTES_5_0_DRIVE_URL}}'
}

# Get direct download link
def get_download_url_from_google_drive_url(google_drive_url):
    if 'drive.google.com' in google_drive_url:
        return f'https://drive.google.com/uc?id={google_drive_url.split("/")[5]}&export=download&confirm=t'
    return google_drive_url

In [ ]:
# Temporary direct download URL placeholders
data_temp_url = {
    '1': '{{PROPBANK_DIRECT_DOWNLOAD_PLACEHOLDER}}',
    '2': '{{ONTONOTES_DIRECT_DOWNLOAD_PLACEHOLDER}}'
}

In [ ]:
for data_name in ['1', '2']:
  google_drive_url = data_name_to_google_drive_url[data_name]
  data_url = get_download_url_from_google_drive_url(google_drive_url)
  print(data_url)
  data_url = data_temp_url[data_name]
  urllib.request.urlretrieve(data_url, "response.zip")
  os.system('unzip response.zip')

In [ ]:
%cd LDC2013T19-ontonotes-release-5.0/data/files/data/english/annotations/nw/wsj

In [ ]:
os.getcwd()

**We follow an approach to map the existing sentences and predicates within the OntoNotes 5.0 Corpus to the sentences and predicates in the extracted dataset (df)**

1st Step: Sentence & Predicate Extraction from OntoNotes 5.0 Corpus

**Annotation Extraction**

In [ ]:
prefix = "wsj"
folder_num = ['00','01','02','03','04','05','06','07','08','09','10','11','12','13','14','15','16','17','18','19','20','21','22','23','24']
file_num = []
for sis in range(1,2455):
  file_num.append(str(f"{sis:04}"))

samp = []
existing_files = []
for folder in folder_num:
  %cd '{folder}'
  path = os.getcwd()
  for sub_file in file_num:
    lines = []
    if(os.path.exists(f'{path}/{prefix}_{sub_file}.prop')):
      with open(f'{prefix}_{sub_file}.prop', 'r') as file:
           existing_files.append(f'{prefix}_{sub_file}.onf')
           data = file.read()
           samp.append(data)
  %cd ..

In [ ]:
samp

In [ ]:
lst = []
for i in range(len(samp)):
  moot = ''
  moot = samp[i].split("\n")
  moot.append('')
  lst.extend(moot)

In [ ]:
lst

In [ ]:
import itertools

w = '' #within sample at the end of each wsj file there is a empety line ('')

split_annots = [list(y) for x, y in itertools.groupby(lst, lambda z: z == w) if not x]

In [ ]:
split_annots

In [ ]:
len (split_annots)

**Index List**

In [ ]:
index = []
for i in range(len(split_annots)):
  lst = []
  for n in range(len(split_annots[i])):
    lst.append(int(split_annots[i][n].split()[1]))
  index.append(list(lst))

In [ ]:
index

**TreeBanked Sentence Extraction**

In [ ]:
prefix = "wsj"
folder_num = ['00','01','02','03','04','05','06','07','08','09','10','11','12','13','14','15','16','17','18','19','20','21','22','23','24']

pmp = []
sent_anchor = []
for folder in folder_num:
  %cd '{folder}'
  path = os.getcwd()
  for file_num in existing_files:
    lines = []
    if(os.path.exists(f'{path}/{file_num}')):
      with open(f'{file_num}', 'r') as file:
           data = file.read()
           lines = data.split('\n\n')
           for i in range(0,len(lines)):
             if (lines[i] == '\n------------------------------------------------------------------------------------------------------------------------' or lines[i] == '------------------------------------------------------------------------------------------------------------------------'):
               sent_anchor.append( (lines[i+2].split("\n--------------------\n    ")[1]).replace("\n   ", "") )
           sent_anchor.append("")
  %cd ..

In [ ]:
import itertools

w = '' #within sample at the end of each wsj file there is a empety line ('')

split_sentences_with_anchor = [list(y) for x, y in itertools.groupby(sent_anchor, lambda z: z == w) if not x]

In [ ]:
split_sentences_with_anchor

**Parse Tree Extraction: For Conversion of Predicates's Annotations into Textual Spans**

In [ ]:
prefix = "wsj"
folder_num = ['00','01','02','03','04','05','06','07','08','09','10','11','12','13','14','15','16','17','18','19','20','21','22','23','24']

pmp = []
tree = []
for folder in folder_num:
  %cd '{folder}'
  path = os.getcwd()
  for file_num in existing_files:
    lines = []
    if(os.path.exists(f'{path}/{file_num}')):
      with open(f'{file_num}', 'r') as file:
           data = file.read()
           lines = data.split('\n\n')
           for i in range(0,len(lines)):
             if (lines[i] == '\n------------------------------------------------------------------------------------------------------------------------' or lines[i] == '------------------------------------------------------------------------------------------------------------------------'):
               tree.append( (lines[i+3].split("Tree:\n-----\n    (TOP ")[1])[:-1] )
           tree.append("")
  %cd ..

In [ ]:
import itertools

w = '' #within sample at the end of each wsj file there is a empety line ('')

split_trees = [list(y) for x, y in itertools.groupby(tree, lambda z: z == w) if not x]

In [ ]:
split_trees

In [ ]:
split_sentences_with_anchor[0][index[0][0]]

In [ ]:
Argument = []
Tree_ = []
Sentence_Anchor = []
for i in range(len(split_annots)):
  for n in range(len(split_annots[i])):
    Argument.append(split_annots[i][n])
    Tree_.append(split_trees[i][index[i][n]])
    Sentence_Anchor.append(split_sentences_with_anchor[i][index[i][n]])

In [ ]:
print(len(Argument))
print(len(Tree_))
print(len(Sentence_Anchor))

In [ ]:
rel = []
for i in range (0,103630):
  rel.append("")

for i in range(len(Argument)):
  list2 = []
  list2.extend(Argument[i].split())
  for s in range(len(list2)):
    if(len(list2[s]) > 4 and list2[s][-1] == "l" and list2[s][-2] == "e" and list2[s][-3] == "r"):
      rel[i] = list2[s].split("-")[0]

In [ ]:
rel

In [ ]:
from nltk.corpus.reader.propbank import PropbankTreePointer
from nltk.tree import Tree

In [ ]:
REL_real = rel.copy()

In [ ]:
def convertor(string):
  delimiters = ["*", ",", ";"]
  for delimiter in delimiters:
    string = " ".join(string.split(delimiter))
  result = string.split()
  return result

In [ ]:
for i in range(len(rel)):
  if (rel[i] == ''):
    REL_real[i] = rel[i]
  else:
    REL_real[i] = convertor(rel[i])

In [ ]:
REL = REL_real.copy()

**Functions**

In [ ]:
def parse_tree_to_sentence(parse_tree_str):
    # Convert string representation of parse tree to NLTK Tree object
    parse_tree = Tree.fromstring(parse_tree_str)

    # Extract words from the parse tree
    words = parse_tree.leaves()

    # Join words to form a sentence
    sentence = ' '.join(words)

    return sentence


In [ ]:
def TreePointer(x):
  lst_2 = []
  lst_2.extend(re.split(';|:', x))
  pointer = PropbankTreePointer(int(lst_2[0]),int(lst_2[1]))
  pointer
  return pointer

In [ ]:
seo = []
for i in range(len(Tree_)):
  if (Tree_[i] == '(S (ADVP (RB Still))\n            (, ,)\n            (NP-SBJ (NNP Westinghouse))\n            (VP (VBZ acknowledges)\n                (SBAR (IN that)\n                      (S (NP-SBJ (NP (NN demand))\n                                 (PP (IN from)\n                                     (NP (JJ independent)\n                                         (NNS producers))))\n                         (VP (MD could)\n                             (VP (VB evaporate)\n                                 (SBAR-ADV (SBAR (IN if)\n                                                 (S (NP-SBJ (NP (NNS prices))\n                                                            (PP (IN for)\n                                                                (NP (NP (NN fuel))\n                                                                    (PP (JJ such)\n                                                                        (IN as)\n                                                                        (NP (NP (JJ natural)\n                                                                                (NN gas))\n                                                                            (CC or)\n                                                                            (NP (NN oil)))))))\n                                                    (VP (VBP rise)\n                                                        (ADVP-MNR (RB sharply)))))\n                                           (CC or)\n                                           (SBAR (IN if)\n                                                 (S (NP-SBJ-3 (NP (NNS utilities))\n                                                              (, ,)\n                                                              (SBAR (WHNP-2 (WDT which))\n                                                                    (S (NP-SBJ-1 (-NONE- *T*-2))\n                                                                       (VP (VBP have)\n                                                                           (VP (VBN been)\n                                                                               (VP (VBN pressured)\n                                                                                   (PP (IN by)\n                                                                                       (NP-LGS (NNS regulators)))\n                                                                                   (NP-4 (-NONE- *-1))\n                                                                                   (S (NP-SBJ (-NONE- *PRO*-4))\n                                                                                      (VP (TO to)\n                                                                                          (VP (VB keep)\n                                                                                              (ADVP-CLR (RB down)))\n                                                                                          (NP (NNS rates))))))))))\n                                                    (, ,))\n                                                 (VP (VBP are)\n                                                     (ADVP-TMP (RB suddenly))\n                                                     (VP (VBN freed)\n                                                         (S (NP-SBJ (-NONE- *PRO*-3))\n                                                            (VP (TO to)\n                                                                (VP (VB add)\n                                                                    (NP (JJ significant)\n                                                                        (VBG generating)\n                                                                        (NN capacity)))))))))))))))\n         (. .)'):
    seo.append(i)

In [ ]:
seo

In [ ]:
s = list(Tree_[23503] + ")")
s[3771] = ''
Tree_[23503] = "".join(s)
Tree_[23504] = "".join(s)
Tree_[23505] = "".join(s)
Tree_[23506] = "".join(s)
Tree_[23507] = "".join(s)
Tree_[23508] = "".join(s)
Tree_[23509] = "".join(s)

In [ ]:
for i in range(len(REL_real)):
  for m in range(len(REL_real[i])):
    if (REL_real[i][m] == ''):
      REL[i][m] = REL_real[i][m]
    else:
      REL[i][m] = parse_tree_to_sentence(str(TreePointer(REL_real[i][m]).select(Tree.fromstring(Tree_[i]))))

In [ ]:
list_sentences_anchor = []
list_predicates = []

for i in range(len(Sentence_Anchor)):
  list_sentences_anchor.append(Sentence_Anchor[i])
  list_predicates.append(' '.join(REL[i]))

In [ ]:
print(len(list_sentences_anchor))
print(len(list_predicates))

In [ ]:
# import pandas as pd
import pandas as pd

# Calling DataFrame constructor after zipping
# both lists, with columns specified
ow = pd.DataFrame(list(zip( list_sentences_anchor, list_predicates)),
               columns =[	'SENTENCE_ANCHOR' ,	'PREDICATES'])

In [ ]:
ow

**Elemination of Trace Anchors**

In [ ]:
import re

def standardize_sentence(sentence):
  pattern = r"0|[*][A-Z]+[*]-\d+|[*][A-Z]+[*]|[*][?][*]|[*]-\d+|[*]"
  cleaned_sentence = re.sub(pattern, '', sentence).strip()
  cleaned_sentence = re.sub(r"\s+", ' ',cleaned_sentence)

  return cleaned_sentence

In [ ]:
sent = []
pred = []
for i in range(0,103630):
  sent.append(standardize_sentence(ow["SENTENCE_ANCHOR"][i]))
  pred.append(standardize_sentence(ow["PREDICATES"][i]))

In [ ]:
data = pd.DataFrame(list(zip(sent , pred)),
               columns =['SENTENCE' ,	'PREDICATES'])

In [ ]:
data

In [ ]:
# df is our extracted dataset, created with contributions from both the PropBank Release and the OntoNotes 5.0 Corpus.

df_list = []

for i in range(0,103748):
  df_list.append([df['SENTENCE'][i] , df['PREDICATES'][i]])

In [ ]:
# data is our standars extracted dataset (within this current notebook), created with OntoNotes 5.0 Corpus.
data_list = []

for i in range(0,103630):
  data_list.append([data['SENTENCE'][i] , data['PREDICATES'][i]])

In [ ]:
df_list

In [ ]:
data_list

**Finding existing indices which related to standard OntoNotes 5.0 Corpus -from our extracted dataset in Notebook (1)- **

In [ ]:
indx = []
for i in range(len(df_list)):
  if df_list[i] in data_list:
    indx.append(i)

In [ ]:
indx

In [ ]:
len(indx)

In [ ]:
dataset = df.iloc[indx].reset_index(drop=True)

In [ ]:
dataset

In [ ]:
from google.colab import files

dataset.to_csv('dataset.csv')
files.download('dataset.csv')